<a href="https://colab.research.google.com/github/pgianinh50364/nlp_course/blob/2024/week02_classification/homework_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Salary prediction, episode II: make it actually work (4 points)

Your main task is to use some of the tricks you've learned on the network and analyze if you can improve __validation MAE__. Try __at least 3 options__ from the list below for a passing grade. Write a short report about what you have tried. More ideas = more bonus points.

__Please be serious:__ " plot learning curves in MAE/epoch, compare models based on optimal performance, test one change at a time. You know the drill :)

You can use either __pytorch__ or __tensorflow__ or any other framework (e.g. pure __keras__). Feel free to adapt the seminar code for your needs. For tensorflow version, consider `seminar_tf2.ipynb` as a starting point.


In [1]:
!pip install lightning
!pip install Keras-Preprocessing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 843.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.3/927.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.3/819.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 981.3 kB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


import gensim.downloader

import torch
from torch import nn
from torch.nn import functional as F
import pytorch_lightning as L
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics.functional import mean_absolute_error
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tokenizers import Tokenizer
from nltk.tokenize import TweetTokenizer
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import warnings
%matplotlib inline
warnings.filterwarnings('ignore')

import gc

In [3]:
data = pd.read_csv('https://raw.githubusercontent.com/pgianinh50364/nlp_course/refs/heads/2024/week02_classification/comments.tsv', sep='\t')

In [4]:
data.head()

,should_ban,comment_text
0,0,The picture on the article is not of the actor...
1,1,"Its madness. Shes of Chinese heritage, but JAP..."
2,1,Fuck You. Why don't you suck a turd out of my ...
3,1,God is dead\nI don't mean to startle anyone bu...
4,1,THIS USER IS A PLANT FROM BRUCE PERENS AND GRO...


In [5]:
print('Average word length of words in data is {0:.0f}.'.format(np.mean(data['comment_text'].apply(lambda x: len(x.split())))))
print('Max word length of words in data is {0:.0f}'.format(np.max(data['comment_text'].apply(lambda x: len(x.split())))))
print('Min word length of words in data is {0:.0f}'.format(np.min(data['comment_text'].apply(lambda x: len(x.split())))))

Average word length of words in data is 60.
Max word length of words in data is 1069
Min word length of words in data is 2


## Feature Engineering

In [6]:
X_train, X_test, y_train, y_test = train_test_split(data['comment_text'], data['should_ban'], test_size=0.1, random_state=42)

In [ ]:
X_train

,comment_text
716,Fanatics on Wikipedia\nDo you dispute the fact...
351,yo man your pissing me off
936,"Come on man, don't be a prick94.195.251.61"
256,I am going to eat your toes.
635,"""\n\n Logo for Simple: \n\nCan you put togethe..."
...,...
106,"""\nThe Graceful Slick....\nIs non other than a..."
270,"""\n\ni find it funny that Mexia, if prounounce..."
860,"fuck you \n\nFuck all the faggot assed, cock s..."
435,"""\n\n""""Bitch, suck my dick before I slap you w..."


In [ ]:
X_test

,comment_text
521,You need a life \n\nREALLY BAD...................
737,"User_talk:TShilo12#Witkacy_WP:POINT Also, Jay..."
740,Your blocks do not deter me \nI may be blocked...
660,"""\n\n More Answers \n\n \nPATTY SAID\n\n<< It..."
411,"GET SOME GLASSES YOU MENTAL FAGOTs, IM Calling..."
...,...
436,"Personally, I'd say, yes it is. In the absence..."
764,no it's actually a fact. look it up fucker.
88,Hilleri groves\nCategory:WelcomeBotResearch\nH...
63,See also section\nI am trying to clean that up...


In [7]:
embeddings = gensim.downloader.load('glove-twitter-200')

[==================================================] 100.0% 758.5/758.5MB downloaded


In [8]:
# Api loading
# import gensim.downloader as api
# glove_model = api.load('glove-twitter-200')

embedding_dim = embeddings.vectors.shape[1]
vocab_size = len(embeddings.key_to_index) + 1
embedding_matrix = np.zeros((vocab_size, embedding_dim))

word_to_idx = {}
for i, word in enumerate(embeddings.key_to_index.keys(), start=1):
  embedding_matrix[i] = embeddings[word]
  word_to_idx[word] = i

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float)


In [23]:
embedding_dim

200

In [18]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

max_len = 200
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_len, padding='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_len, padding='post')

X_train_tensor = torch.stack([torch.from_numpy(seq.astype(np.int32)) for seq in X_train_padded]).float()
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.stack([torch.from_numpy(seq.astype(np.int32)) for seq in X_test_padded]).float()
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Tokenized and padded input:\n", X_train_padded)
print("Word index:\n", tokenizer.word_index)

Tokenized and padded input:
 [[3396   22   41 ...    0    0    0]
 [1350  329   18 ...    0    0    0]
 [ 280   22  329 ...    0    0    0]
 ...
 [  65    3   65 ...    0    0    0]
 [ 130   13   25 ...    3   27   12]
 [  41  281   54 ...    0    0    0]]
Word index:
 {'<OOV>': 1, 'the': 2, 'you': 3, 'to': 4, 'a': 5, 'and': 6, 'i': 7, 'of': 8, 'is': 9, 'that': 10, 'in': 11, 'it': 12, 'suck': 13, 'this': 14, 'are': 15, 'for': 16, 'not': 17, 'your': 18, 'be': 19, 'hitler': 20, 'heil': 21, 'on': 22, 'offfuck': 23, 'as': 24, 'my': 25, 'have': 26, 'with': 27, 'do': 28, 'me': 29, 'if': 30, 'or': 31, 'but': 32, 'so': 33, 'page': 34, 'article': 35, 'what': 36, 'was': 37, 'an': 38, 'know': 39, 'all': 40, 'wikipedia': 41, 'nigger': 42, 'j': 43, 'delanoy': 44, 'at': 45, 'from': 46, 'talk': 47, 'dick': 48, 'because': 49, 'can': 50, 'no': 51, 'they': 52, 'by': 53, 'like': 54, 'will': 55, "don't": 56, 'one': 57, 'who': 58, 'about': 59, 'just': 60, 'he': 61, 'there': 62, 'old': 63, 'has': 64, 'fuck'

In [28]:
class MyCNN(L.LightningModule):
  def __init__(self, input_dims, hidden_dims, drop_out, kernel_size, learning_rate=1e-3):
    super().__init__()
    self.input_dims = input_dims
    self.hidden_dims = hidden_dims
    self.kernel_size = kernel_size
    self.drop_out = drop_out
    self.learning_rate = learning_rate

    self.conv1d = nn.Conv1d(input_dims, hidden_dims, kernel_size=kernel_size)
    self.batchnorm = nn.BatchNorm1d(input_dims)
    self.dropout = nn.Dropout(drop_out)

  def forward(self, x):
    x = self.conv1d(x)
    x = self.batchnorm(x)
    x = self.dropout(x)

  def training_step(self, batch, batch_idx):
    x, y = batch
    y_hat = self.forward(x)
    loss = F.cross_entropy(y_hat, y)
    y_pred = torch.argmax(y_hat, dim=1)
    mae = mean_absolute_error(y_pred.float(), y.float())
    self.log("train_loss", loss, prog_bar=True)
    self.log("MAE", mae, prog_bar=True)
    return loss

  def validation_step(self, batch, batch_idx):
    x, y = batch
    y_hat = self(x)
    loss = F.cross_entropy(y_hat, y)
    y_pred = torch.argmax(y_hat, dim=1)
    mae = mean_absolute_error(y_pred.float(), y.float())
    self.log("val_loss", loss, prog_bar=True)
    return x

  def configure_optimizers(self):
    optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
    return optimizer

In [ ]:
'''
class MyCNN(L.LightningModule):
  def __init__(self, input_dims, hidden_dims, drop_out, kernel_size):
    super().__init__()
    self.input_dims = input_dims
    self.hidden_dims = hidden_dims
    self.kernel_size = kernel_size
    self.drop_out = drop_out

    self.conv1d = nn.Conv1d(input_dims, hidden_dims, kernel_size=kernel_size)
    self.batchnorm = nn.BatchNorm1d(hidden_dims)
    self.dropout = nn.Dropout(drop_out)

  def forward(self, x):
    x = self.conv1d(x)
    x = self.batchnorm(x)
    x = self.dropout(x)
    return x
'''
class MyBiLSTM(L.LightningModule):
  def __init__(self, input_dims, hidden_dims):
    super().__init__()
    self.input_dims = input_dims
    self.hidden_dims = hidden_dims

    self.lstm = nn.LSTM(input_dims, hidden_dims, bidirectional=True, batch_first=True)

  def forward(self, x):
    if x.shape[-1] == self.input_dims:
      x = x.transpose(-1, -2)
    self.lstm.flatten_parameters()
    x, _ = self.lstm(x)
    return x

class AttnPooling(L.LightningModule):
  def __init__(self, input_dims, seq_len):
    super().__init__()
    self.input_dims = input_dims
    self.seq_len = seq_len
    self.attn = nn.Linear(input_dims * seq_len, 1)  # Outputs scalar attention scores for each feature vector.

  def forward(self, batch):
    # Input: batch (B, T, D)
    if len(batch.shape) == 2:
      batch = batch.unsqueeze(1)

    attn_scores = self.attn(batch)  # Apply attention layer to each time step, shape: (B, T, 1)
    attn_weights = F.softmax(attn_scores, dim=1)  # Apply softmax over time steps
    output = torch.sum(batch * attn_weights, dim=1)  # Weighted sum over time steps
    return output
    return output

In [ ]:
class CNNwEmbeddings(L.LightningModule):
  def __init__(self, embedding_matrix, hidden_dims, num_classes, input_length, drop_out, learning_rate=1e-3):
      super().__init__()
      self.save_hyperparameters()
      self.hidden_dims = hidden_dims
      self.num_classes = num_classes
      self.drop_out = drop_out
      self.input_length = input_length
      # Pre-trainded embeddings
      vocab_size, embedding_dims = embedding_matrix.size()
      self.embedding = nn.Embedding(vocab_size, embedding_dim)
      self.embedding.weight = nn.Parameter(embedding_matrix, requires_grad=False)
      # Convo
      self.conv1 = nn.Conv1d(embedding_dim, hidden_dims, kernel_size=3)
      self.bn1 = nn.BatchNorm1d(hidden_dims)
      self.dropout1 = nn.Dropout(drop_out)
      # Convo 2
      self.conv2 = nn.Conv1d(hidden_dims, hidden_dims//2, kernel_size=3)
      self.bn2 = nn.BatchNorm1d(hidden_dims//2)
      self.dropout2 = nn.Dropout(drop_out)
      # Pooling
      self.attnpool = AttnPooling(hidden_dims//2, input_length - 4)
      # Fully connected
      self.fc = nn.Linear(hidden_dims//2, num_classes)
      self.learning_rate = learning_rate

  def forward(self, x):
      x = self.embedding(x)
      x = x.permute(0, 2, 1)
      x = F.relu(self.conv1(x))
      x = self.bn1(x)
      x = self.dropout1(x)
      x = F.relu(self.conv2(x))
      x = self.bn2(x)
      x = self.dropout2(x)
      x = self.attnpool(x)
      x = self.fc(x)
      return x

  def training_step(self, batch, batch_idx):
      x, y = batch
      y_hat = self.forward(x)
      loss = F.cross_entropy(y_hat, y)
      y_pred = torch.argmax(y_hat, dim=1)
      mae = mean_absolute_error(y_pred.float(), y.float())
      self.log("train_loss", loss, prog_bar=True)
      self.log("MAE", mae, prog_bar=True)
      return loss

  def validation_step(self, batch, batch_idx):
      x, y = batch
      y_hat = self(x)
      loss = F.cross_entropy(y_hat, y)
      y_pred = torch.argmax(y_hat, dim=1)
      mae = mean_absolute_error(y_pred.float(), y.float())
      self.log("val_loss", loss, prog_bar=True)
      self.log("MAE", mae, prog_bar=True)
      return loss

  def configure_optimizers(self):
      optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
      return optimizer


In [31]:
model = MyCNN(embedding_dim, 128, .2, 3)

In [32]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",  # Metric to monitor
    dirpath="/content/cktpt",  # Directory to save checkpoints
    filename="cnn_model_ver1",  # Filename for the best model
    save_top_k=1,  # Save only the top model
    mode="min"  # Minimize the monitored metric
)
early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min"
)

# Define trainer
trainer = L.Trainer(
    max_epochs=20,
    callbacks=[checkpoint_callback, early_stopping_callback],
    accelerator="cuda",
    devices=-1
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [33]:
trainer.fit(model, train_dataloaders=train_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type        | Params | Mode 
--------------------------------------------------
0 | conv1d    | Conv1d      | 76.9 K | train
1 | batchnorm | BatchNorm1d | 400    | train
2 | dropout   | Dropout     | 0      | train
--------------------------------------------------
77.3 K    Trainable params
0         Non-trainable params
77.3 K    Total params
0.309     Total estimated model params size (MB)
3         Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

RuntimeError: Given groups=1, weight of size [128, 200, 3], expected input[1, 32, 200] to have 200 channels, but got 32 channels instead

In [24]:
num_classes = len(set(y_train))
num_classes

2

In [ ]:
trainer.validate(model, dataloaders=test_loader)

In [ ]:
# Extract logged metrics
train_mae = trainer.logged_metrics.get('train_mae')
val_mae = trainer.logged_metrics.get('val_mae')

# Create a plot
plt.figure(figsize=(10, 6))
plt.plot(train_mae, label='Training MAE', color='blue')
plt.plot(val_mae, label='Validation MAE', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Mean Absolute Error (MAE)')
plt.title('Training and Validation MAE')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Predicting new classes (if we have new data)
model.eval()
with torch.no_grad():
    predictions = model(new_data)
    predicted_classes = torch.argmax(predictions, dim=1)
    print("Predicted Classes:", predicted_classes)

### A short report

Please tell us what you did and how did it work.

`<YOUR_TEXT_HERE>`, i guess...

## Recommended options

#### A) CNN architecture

All the tricks you know about dense and convolutional neural networks apply here as well.
* Dropout. Nuff said.
* Batch Norm. This time it's `nn.BatchNorm*`/`L.BatchNormalization`
* Parallel convolution layers. The idea is that you apply several nn.Conv1d to the same embeddings and concatenate output channels.
* More layers, more neurons, ya know...


#### B) Play with pooling

There's more than one way to perform pooling:
* Max over time (independently for each feature)
* Average over time (excluding PAD)
* Softmax-pooling:
$$ out_{i, t} = \sum_t {h_{i,t} \cdot {{e ^ {h_{i, t}}} \over \sum_\tau e ^ {h_{j, \tau}} } }$$

* Attentive pooling
$$ out_{i, t} = \sum_t {h_{i,t} \cdot Attn(h_t)}$$

, where $$ Attn(h_t) = {{e ^ {NN_{attn}(h_t)}} \over \sum_\tau e ^ {NN_{attn}(h_\tau)}}  $$
and $NN_{attn}$ is a dense layer.

The optimal score is usually achieved by concatenating several different poolings, including several attentive pooling with different $NN_{attn}$ (aka multi-headed attention).

The catch is that keras layers do not inlude those toys. You will have to [write your own keras layer](https://keras.io/layers/writing-your-own-keras-layers/). Or use pure tensorflow, it might even be easier :)

#### C) Fun with words

It's not always a good idea to train embeddings from scratch. Here's a few tricks:

* Use a pre-trained embeddings from `gensim.downloader.load`. See last lecture.
* Start with pre-trained embeddings, then fine-tune them with gradient descent. You may or may not download pre-trained embeddings from [here](http://nlp.stanford.edu/data/glove.6B.zip) and follow this [manual](https://keras.io/examples/nlp/pretrained_word_embeddings/) to initialize your Keras embedding layer with downloaded weights.
* Use the same embedding matrix in title and desc vectorizer


#### D) Going recurrent

We've already learned that recurrent networks can do cool stuff in sequence modelling. Turns out, they're not useless for classification as well. With some tricks of course..

* Like convolutional layers, LSTM should be pooled into a fixed-size vector with some of the poolings.
* Since you know all the text in advance, use bidirectional RNN
  * Run one LSTM from left to right
  * Run another in parallel from right to left
  * Concatenate their output sequences along unit axis (dim=-1)

* It might be good idea to mix convolutions and recurrent layers differently for title and description


#### E) Optimizing seriously

* You don't necessarily need 100 epochs. Use early stopping. If you've never done this before, take a look at [early stopping callback(keras)](https://keras.io/callbacks/#earlystopping) or in [pytorch(lightning)](https://pytorch-lightning.readthedocs.io/en/latest/common/early_stopping.html).
  * In short, train until you notice that validation
  * Maintain the best-on-validation snapshot via `model.save(file_name)`
  * Plotting learning curves is usually a good idea
  
Good luck! And may the force be with you!